# Block T trên Kaggle: teacher 32B → student 4B/8B/14B

Bật **GPU T4 x2** và **Internet**. Vào **Add-ons > Secrets**, tạo `HF_TOKEN` (token có quyền ghi).
Mỗi tài khoản Kaggle mở notebook này, chỉ sửa ô cấu hình rồi Run All.

| Bước | `STAGE` | Chạy ở đâu | Ghi chú |
| --- | --- | --- | --- |
| T2 | `generate` | mỗi tài khoản một `SHARD` | teacher viết query |
| T3 | (laptop) | `teach.py mine` | cần index Block B |
| T4 | `judge` | mỗi tài khoản một `SHARD` | teacher chấm cặp |
| T5 | `train` | 2 student song song, mỗi T4 một student | |

Session bị dừng giữa chừng thì Run All lại với cùng cấu hình: kết quả đã lưu trên Hub được tải về và chạy tiếp.
Chi tiết: `docs/solution/teaching_guide.md`.

In [ ]:
STAGE = 'generate'          # 'generate' | 'judge' | 'train'
SHARD = 0                   # tài khoản thứ mấy: 0..SHARDS-1 (generate/judge)
SHARDS = 4                  # phải khớp --shards lúc sample/mine
HUB_REPO = 'AncientWings/vibiomir-teach'          # HF dataset private: code, input, kết quả teacher
HUB_MODEL_REPO = 'AncientWings/vibiomir-students' # HF model repo private: adapter student
TEACHER = 'qwen3-32b-awq'   # vLLM; nếu vLLM không chạy được trên T4: 'qwen3-32b' + BACKEND='hf'
BACKEND = None
HOURS = 11.0                # dừng sạch trước giới hạn 12 giờ
# T5: tối đa 2 job, mỗi job một GPU. (task, size): task in reranker|embedding|translator
TRAIN_JOBS = [('reranker', '4b'), ('embedding', '4b')]
RESUME = True
TRAIN_LIMIT = None          # vd. 2000 để chạy thử nhanh

In [ ]:
import os, subprocess, sys
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
if STAGE in ('generate', 'judge') and (BACKEND or 'vllm') == 'vllm':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft', 'bitsandbytes',
                    'accelerate'], check=True)
import torch
print(torch.__version__, torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
from pathlib import Path
import shutil
from huggingface_hub import snapshot_download

BASE = Path('/kaggle/working/vibiomir')
patterns = ['code/**', 'inputs/**'] + (['teach/**'] if STAGE == 'train' else [])
snapshot_download(HUB_REPO, repo_type='dataset', local_dir=BASE, allow_patterns=patterns,
                  token=os.environ['HF_TOKEN'])
TEACH = BASE / 'teach'
TEACH.mkdir(exist_ok=True)
if (BASE / 'inputs' / 'groups.jsonl').exists():
    shutil.copy(BASE / 'inputs' / 'groups.jsonl', TEACH / 'groups.jsonl')
print(sorted(str(p.relative_to(BASE)) for p in BASE.rglob('*') if p.is_file() and 'code' not in p.parts))

In [ ]:
SCRIPT = str(BASE / 'code' / 'scripts' / 'teach.py')
INPUTS = BASE / 'inputs'

def launch(args, gpu=None):
    env = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}
    if gpu is not None:
        env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    return subprocess.Popen([sys.executable, '-u', SCRIPT, *map(str, args)], env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

def follow(processes):
    import selectors
    selector = selectors.DefaultSelector()
    for label, process in processes:
        selector.register(process.stdout, selectors.EVENT_READ, label)
    open_streams = len(processes)
    while open_streams:
        for key, _ in selector.select():
            line = key.fileobj.readline()
            if not line:
                selector.unregister(key.fileobj)
                open_streams -= 1
                continue
            print(f'[{key.data}] {line}', end='', flush=True)
    codes = {label: process.wait() for label, process in processes}
    print(codes)
    if any(codes.values()):
        raise RuntimeError(f'Có tiến trình lỗi: {codes}')

common = ['--out', TEACH, '--shard', SHARD, '--shards', SHARDS, '--teacher', TEACHER,
          '--hub-repo', HUB_REPO, '--hours', HOURS] + (['--backend', BACKEND] if BACKEND else [])
if STAGE == 'generate':
    follow([('generate', launch(['generate', '--seeds', INPUTS / 'seeds.jsonl', *common]))])
elif STAGE == 'judge':
    follow([('judge', launch(['judge', '--groups', TEACH / 'groups.jsonl',
                               '--chunks', INPUTS / 'chunks.parquet', *common]))])
elif STAGE == 'train':
    if len(TRAIN_JOBS) > torch.cuda.device_count():
        raise ValueError('Mỗi job cần một GPU riêng.')
    jobs = []
    for gpu, (task, size) in enumerate(TRAIN_JOBS):
        args = ['train', '--task', task, '--size', size, '--teach', TEACH,
                '--chunks', INPUTS / 'chunks.parquet', '--out', '/kaggle/working/students',
                '--hub-repo', HUB_MODEL_REPO, '--hours', HOURS]
        if RESUME:
            args.append('--resume')
        if TRAIN_LIMIT:
            args += ['--limit', TRAIN_LIMIT]
        jobs.append((f'{task}-{size}', launch(args, gpu=gpu)))
    follow(jobs)